# Seguimiento completo del pipeline

Este notebook documenta las tres fases del modelo de machine learning:

1. **Entrenamiento** — el modelo aprende patrones a partir de datos históricos etiquetados (RECIBIDO = 0, ABANDONO/BAJA = 1)
2. **Validación** — se evalúa el rendimiento del modelo sobre datos que no vio durante el entrenamiento
3. **Pronóstico** — el modelo ya entrenado predice el riesgo de abandono sobre estudiantes activos cuyo desenlace aún se desconoce (CURSO/PAUSA)

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd  # noqa: F401

sys.path.append(os.path.abspath(os.path.join('..')))

from src.ingestor_datos import IngestorDatos
from src.preparador_datos import PreparadorDatos
from src.entrenador_modelos import EntrenadorModelos
from src.evaluador_riesgo import EvaluadorRiesgo

---
## Fase 1 — Entrenamiento (datos etiquetados)

Los **datos etiquetados** son aquellos cuyo estado académico final ya se conoce:
- `target_ml = 0` → estudiante **RECIBIDO** (continúa)
- `target_ml = 1` → estudiante **ABANDONO** o **BAJA** (no continúa)

Con estos datos el modelo aprende a reconocer qué patrones de notas, asistencia y datos sociodemográficos se asocian al abandono.

In [2]:
# Carga de datos
ingestor = IngestorDatos()
datasets = ingestor.leer_datos()

# Preparación
preparador = PreparadorDatos(datasets)
df_master = preparador.ejecutar_preparacion()

# Dataset para ML
df_ml = preparador.preparar_dataset_ml(df_master)

# Filtrar solo datos etiquetados (target conocido)
df_etiquetados = df_ml.dropna(subset=['target_ml'])
total_etiquetados = len(df_etiquetados)
abandonos = int(df_etiquetados['target_ml'].sum())
continuan = total_etiquetados - abandonos

print('=== FASE 1: ENTRENAMIENTO (datos etiquetados) ===')
print(f'Total registros etiquetados: {total_etiquetados}')
print(f'  - Continúa (target=0): {continuan}')
print(f'  - Abandono (target=1): {abandonos}')
print(f'  - Proporción abandono: {abandonos/total_etiquetados:.1%}')
print()
print('Columnas predictoras disponibles para entrenar:')
print(list(df_etiquetados.drop(columns=['target_ml']).columns))

Cargando inscripciones desde LSE_Inscrip_Baja_Recibido.csv...
Cargando notas_bimestre desde LSE_Notas_Estadistica_Bimestre.csv...
Cargando actual desde LSE_Notas_Inscrip_Baja_Actual.xlsx...
Preprocesamiento: Se eliminaron 126 registros duplicados en la base.
Tabla Maestra generada con 646 registros y nulos tratados.
=== FASE 1: ENTRENAMIENTO (datos etiquetados) ===
Total registros etiquetados: 95
  - Continúa (target=0): 62
  - Abandono (target=1): 33
  - Proporción abandono: 34.7%

Columnas predictoras disponibles para entrenar:
['sd_nota', 'sd_asist', 'edad', 'postgrado', 'unnamed: 28', 'asist_b1', 'asist_b2', 'asist_b3', 'asist_b4', 'asist_b5', 'asist_b6', 'asist_b7', 'asist_b8', 'asist_b9', 'asist_b10', 'nota_b1', 'nota_b2', 'nota_b3', 'nota_b4', 'nota_b5', 'nota_b6', 'nota_b7', 'nota_b8', 'nota_b9', 'nota_b10', 'estudio_CEIOT', 'estudio_CESE', 'estudio_MSE', 'pais_OTRO PAIS', 'provincia_CABA (CABA)', 'provincia_Catamarca', 'provincia_Chaco', 'provincia_Chubut', 'provincia_Corrient

---
## Fase 2 — Validación (métricas por bimestre)

Para cada bimestre (del 1 al 6) se entrena un árbol de decisión independiente, usando solo las columnas de notas y asistencia disponibles hasta ese hito temporal. Se divide el conjunto etiquetado en:
- **80% entrenamiento** — el modelo aprende
- **20% prueba** — el modelo se evalúa con datos que no ha visto

Las métricas principales son:
- **Accuracy**: proporción de aciertos global
- **Recall (Abandono)**: proporción de abandonos reales que el modelo detecta correctamente. Es la métrica de negocio más importante porque buscamos minimizar los falsos negativos.

In [3]:
# Entrenamiento con validación
entrenador = EntrenadorModelos()
df_metricas = entrenador.entrenar_y_guardar(df_ml, preparador)

print('=== FASE 2: VALIDACIÓN ===')
print()
print(df_metricas.to_string(index=False))
print()
print(f'Accuracy promedio: {df_metricas["Accuracy"].mean():.2f}')
print(f'Recall (Abandono) promedio: {df_metricas["Recall (Abandono)"][df_metricas["Recall (Abandono)"] > 0].mean():.2f}')


 INICIANDO ENTRENAMIENTO POR PUNTOS DE CONTROL (XAI)
Proceso de entrenamiento completado. Modelos guardados en disco.

=== FASE 2: VALIDACIÓN ===

      Hito  Registros etiquetados  Train (80%)  Test (20%)  Abandono en train  Continúa en train  Accuracy  Recall (Abandono) Variable Clave  Importancia
Bimestre 1                     95           76          19                 27                 49      0.53               0.67        nota_b1         0.22
Bimestre 2                     95           76          19                 27                 49      0.74               1.00        nota_b2         0.29
Bimestre 3                     95           76          19                 27                 49      0.68               0.67        nota_b2         0.31
Bimestre 4                     95           76          19                 27                 49      0.84               0.83        nota_b4         0.43
Bimestre 5                     95           76          19                 27     

---
## Fase 3 — Pronóstico (datos sin etiqueta)

Los **datos sin etiqueta** corresponden a los estudiantes actualmente activos en estado **CURSO** o **PAUSA**, cuyo desenlace final (graduación o abandono) aún se desconoce.

El modelo entrenado (árbol de decisión) infiere su probabilidad de abandono basándose en los patrones aprendidos en la Fase 1. Los umbrales de decisión son:
- `probabilidad >= 0.70` → **ALTO**
- `probabilidad >= 0.40` → **MEDIO**
- `probabilidad < 0.40` → **BAJO**

In [4]:
# Evaluación con el modelo guardado en disco
evaluador = EvaluadorRiesgo()
df_riesgo = evaluador.ejecutar_evaluacion(df_master)

print('=== FASE 3: PRONÓSTICO (datos sin etiqueta) ===')
print()

# Separar por tipo de predicción
df_pronostico = df_riesgo[df_riesgo['tipo_prediccion'] == 'PRONÓSTICO']
df_historico = df_riesgo[df_riesgo['tipo_prediccion'] == 'HISTÓRICO']

print(f'Alumnos históricos (target conocido): {len(df_historico)}')
print(f'Alumnos pronosticados (sin etiqueta): {len(df_pronostico)}')
print()

if len(df_pronostico) > 0:
    print('Distribución del riesgo pronosticado:')
    for nivel in ['ALTO', 'MEDIO', 'BAJO']:
        conteo = len(df_pronostico[df_pronostico['nivel_riesgo'] == nivel])
        print(f'  {nivel}: {conteo}')
    print()

# Mostrar alumnos con riesgo alto
alumnos_alto = df_pronostico[df_pronostico['nivel_riesgo'] == 'ALTO']
if len(alumnos_alto) > 0:
    print('Alumnos en riesgo ALTO (justificación XAI):')
    cols_show = ['n_siu', 'estudio', 'probabilidad_abandono', 'justificacion_riesgo']
    print(alumnos_alto[cols_show].to_string(index=False))

=== FASE 3: PRONÓSTICO (datos sin etiqueta) ===

Alumnos históricos (target conocido): 95
Alumnos pronosticados (sin etiqueta): 79

Distribución del riesgo pronosticado:
  ALTO: 73
  MEDIO: 1
  BAJO: 5

Alumnos en riesgo ALTO (justificación XAI):
  n_siu estudio  probabilidad_abandono                                                                                            justificacion_riesgo
2966003     MSE               1.000000   [Hito B1] nota_b1 > 6.50 AND sd_nota <= 0.56 AND provincia_Santa Fe <= 0.50 AND provincia_CABA (CABA) <= 0.50
  E1411    CESE               1.000000                                                                                       [Hito B1] nota_b1 <= 6.50
  E1214     MSE               1.000000                                                                                       [Hito B1] nota_b1 <= 6.50
  E1615    CESE               1.000000                                                                                       [Hito B1] nota_b1 <= 6.5

---
## Resumen del pipeline completo

Se han ejecutado las tres fases del modelo de machine learning:

| Fase | Datos | Target | Salida |
|---|---|---|---|
| 1. Entrenamiento | Históricos (RECIBIDO/BAJA) | Conocido (0/1) | Modelos entrenados por bimestre |
| 2. Validación | 20% de los históricos | Conocido (0/1) | Métricas (Accuracy, Recall) |
| 3. Pronóstico | Activos (CURSO/PAUSA) | Desconocido | Riesgo inferido + Justificación XAI |

Este diseño garantiza que los datos de pronóstico **nunca se mezclan** con los datos de entrenamiento, preservando la pureza experimental.